# Consensus Industry Classification — Modeling Experiment

**Purpose:** Compare the Worth AI production industry classification pipeline against the new Consensus XGBoost approach on the same data.

| | Scenario A — Production | Scenario B — Consensus XGBoost |
|---|---|---|
| Logic | `customer_table.sql`: `max(zi_conf, efx_conf)` wins | 45-feature `multi:softprob` XGBoost |
| Model | No ML for classification — rule only | Level 2 XGBoost (trained here) |
| Output | 1 NAICS code, no probability | Probability + top-3 + UK SIC + AML + KYB |
| UK SIC | Available from OC — **silently dropped** | Routed as primary for GB companies |
| AML / KYB | Not produced | 5 AML signal types + APPROVE/ESCALATE/REJECT |

**Data source:** `datascience.customer_files` + match tables (Redshift), or realistic synthetic mirror when Redshift is unavailable.

**Before running:** Execute `python modeling/experiment.py --mode full` to generate artifacts, then run all cells.

In [ ]:
import sys, json, warnings, logging
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter

from modeling.config import ARTIFACTS, ALL_FEATURES
from modeling.data_loader import DataLoader
from modeling.level1_evaluator import Level1Evaluator
from modeling.production_baseline import ProductionBaseline
from modeling.level2_trainer import Level2Trainer
from modeling.comparison import ComparisonEngine

# Dark theme
plt.rcParams.update({
    'figure.facecolor': '#0F1117', 'axes.facecolor': '#1A1F2E',
    'axes.edgecolor': '#2D3748', 'axes.labelcolor': '#E2E8F0',
    'xtick.color': '#A0AEC0', 'ytick.color': '#A0AEC0',
    'text.color': '#E2E8F0', 'grid.color': '#2D3748',
    'grid.alpha': 0.5, 'font.size': 11,
})
BLUE='#4299E1'; GREEN='#48BB78'; RED='#FC8181'; GOLD='#ECC94B'; GRAY='#718096'

# Load outputs from experiment.py
comp_df    = pd.read_parquet(ARTIFACTS['comparison_report'])
eval_report = json.load(open(ARTIFACTS['evaluation_report']))
n = len(comp_df)

print(f"Loaded comparison report: {n:,} companies")
print(f"Data source: {eval_report.get('data_source','unknown')}")
print(f"Unique NAICS classes: {eval_report.get('n_classes','—')}")
print(f"Training rows: {eval_report.get('n_train','—')} | Test rows: {eval_report.get('n_test','—')}")

## 1. Data Source and Quality

In [ ]:
loader = DataLoader()
print(f"Redshift available:       {loader.redshift_available}")
print(f"Case-service PG available:{loader.caseservice_available}")
print()

# Load raw data for analysis
raw_df = loader.load_features(limit=5000)
print(f"Raw data shape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns[:10])} ...")
print()
print("Confidence score ranges:")
for col in ['oc_confidence','efx_confidence','zi_confidence','liberty_confidence']:
    if col in raw_df.columns:
        c = pd.to_numeric(raw_df[col], errors='coerce')
        print(f"  {col:25s}: min={c.min():.3f}  mean={c.mean():.3f}  max={c.max():.3f}")

## 2. Level 1 Entity Matching — Output Analysis

The Level 1 model (`entity_matching_20250127 v1`) already exists in production.
Here we analyse its outputs: match rates, confidence distributions, and source agreement.

In [ ]:
ev1 = Level1Evaluator()
l1_report = ev1.evaluate(raw_df, ground_truth_col='label_naics')
summary_df = ev1.summary_table(raw_df)
display(summary_df)

In [ ]:
# Confidence distribution per source
conf_long = ev1.confidence_distribution_df(raw_df)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=False)
fig.suptitle('Level 1 Match Confidence Distributions per Source', fontsize=13, fontweight='bold')
colours = {'OC': BLUE, 'EFX': GREEN, 'ZI': GOLD, 'LIBERTY': RED}

for ax, src in zip(axes, ['OC','EFX','ZI','LIBERTY']):
    data = conf_long[conf_long['source'] == src]['confidence']
    if data.empty:
        ax.set_title(f'{src}\n(no data)')
        continue
    ax.hist(data, bins=30, color=colours[src], alpha=0.85, edgecolor='#0F1117')
    ax.axvline(0.80, color='white', linestyle='--', linewidth=1.5, label='Threshold 0.80')
    ax.set_title(f'{src}\nmatch rate: {(data>=0.80).mean():.0%}', fontweight='bold')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Companies')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('data/modeling/l1_confidence_dist.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

print("\nNAICS accuracy when source matched at >= 0.80:")
for src, stats in l1_report.get('naics_accuracy_when_matched', {}).items():
    print(f"  {src.upper():10s}: exact={stats['top1_exact_pct']}%  4-digit={stats['top1_4digit_pct']}%")

## 3. Scenario A — Production Baseline

Applying `customer_table.sql` logic: `IF zi_conf > efx_conf THEN zi_naics ELSE efx_naics`

In [ ]:
baseline = ProductionBaseline()
df_a = baseline.run(raw_df)
eval_a = baseline.evaluate(df_a, ground_truth_col='label_naics')

print("Production Baseline Evaluation:")
print(f"  Coverage:            {eval_a['coverage_pct']}%")
print(f"  Top-1 accuracy:      {eval_a['top1_accuracy_pct']}%")
print(f"  Top-1 (4-digit):     {eval_a['top1_4digit_pct']}%")
print(f"  ZoomInfo wins:       {eval_a['zoominfo_wins']:,}")
print(f"  Equifax wins:        {eval_a['equifax_wins']:,}")
print(f"  UK SIC available:    {eval_a['uk_sic_available']:,}")
print(f"  UK SIC stored:       {eval_a['uk_sic_stored']}  ← THE GAP")
print(f"  OC conflicts winner: {eval_a['oc_conflicts_with_winner']:,}")
print(f"  AML signals:         {eval_a['aml_signals']}")
print(f"  KYB produced:        {eval_a['kyb_produced']}")
print()
print(f"  ⚠️  {eval_a['notes']}")

In [ ]:
# ZoomInfo vs Equifax winner split
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Scenario A — Production Baseline', fontsize=13, fontweight='bold')

ax = axes[0]
zi_w = eval_a['zoominfo_wins']; efx_w = eval_a['equifax_wins']
ax.pie([zi_w, efx_w], labels=[f'ZoomInfo\n{zi_w:,}', f'Equifax\n{efx_w:,}'],
       colors=[BLUE, GREEN], startangle=90,
       autopct='%1.1f%%', textprops={'color':'#E2E8F0'})
ax.set_title('Winner-takes-all\n(which source wins per company)', fontweight='bold')

ax = axes[1]
uk_a = eval_a['uk_sic_available']; uk_s = 0; uk_r = n - uk_a
ax.pie([uk_s, uk_a - uk_s, uk_r],
       labels=[f'Stored to DB\n{uk_s}', f'Available but DROPPED\n{uk_a}', f'Not in OC\n{uk_r}'],
       colors=[GREEN, RED, GRAY], startangle=90,
       autopct=lambda p: f'{p:.1f}%' if p > 1 else '',
       textprops={'color':'#E2E8F0'})
ax.set_title('UK SIC from OpenCorporates\n(available vs stored)', fontweight='bold')

plt.tight_layout()
plt.savefig('data/modeling/scenario_a.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 4. Level 2 Training — Consensus XGBoost

The model is trained on the 45-feature vector using the labels loaded by `data_loader.py`.
If `experiment.py` was already run, we load the saved artifacts. Otherwise we retrain here.

In [ ]:
print("Level 2 Evaluation Report:")
print(f"  Training rows:   {eval_report.get('n_train',0):,}")
print(f"  Test rows:       {eval_report.get('n_test',0):,}")
print(f"  NAICS classes:   {eval_report.get('n_classes',0)}")
print(f"  Top-1 accuracy:  {eval_report.get('top1_accuracy_pct','—')}%")
print(f"  Top-3 accuracy:  {eval_report.get('top3_accuracy_pct','—')}%")
print(f"  Log-loss:        {eval_report.get('log_loss','—')}")
print()
print("Top-10 features by importance:")
for feat, imp in eval_report.get('top10_features', {}).items():
    bar = '█' * int(imp * 1000)
    print(f"  {feat:35s} {imp:.5f}  {bar}")

In [ ]:
# Feature importance chart
fi = eval_report.get('top10_features', {})
feat_labels = {
    'oc_matched':            'OpenCorporates matched (≥0.80)',
    'zi_matched':            'ZoomInfo matched (≥0.80)',
    'efx_matched':           'Equifax matched (≥0.80)',
    'hi_risk_sector':        'Code in AML high-risk sector',
    'source_majority_agreement': 'Source majority agreement',
    'web_registry_distance': 'Web vs registry distance',
    'temporal_pivot_score':  'Temporal pivot score',
    'avg_confidence':        'Avg confidence across sources',
    'is_naics_jurisdiction': 'NAICS-primary jurisdiction',
    'j_us_state':            'US state jurisdiction',
    'oc_confidence':         'OC weighted confidence',
    'zi_confidence':         'ZI weighted confidence',
    'sources_above_threshold': 'Sources above 50% threshold',
    'max_confidence':        'Max single-source confidence',
}

fig, ax = plt.subplots(figsize=(13, 5))
names  = [feat_labels.get(k, k) for k in fi.keys()]
values = list(fi.values())
clrs   = [RED if any(w in k for w in ['risk','pivot','dist','pollut'])
          else GREEN for k in fi.keys()]

bars = ax.barh(list(range(len(names)))[::-1], values, color=clrs, alpha=0.85)
ax.set_yticks(list(range(len(names))))
ax.set_yticklabels(names[::-1], fontsize=10)
ax.set_title('Feature Importances — Consensus XGBoost Level 2\n'
             '(red = AML/risk features, green = classification quality)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('data/modeling/feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# Per-sector accuracy
per_sector = eval_report.get('per_sector', {})
if per_sector:
    sec_df = pd.DataFrame([
        {'Sector': k, 'N': v['n'], 'Top-1 %': v['top1_pct']}
        for k, v in per_sector.items()
    ]).sort_values('Top-1 %', ascending=False)

    fig, ax = plt.subplots(figsize=(13, 4))
    clrs_s = [GREEN if v >= 50 else GOLD if v >= 25 else RED
              for v in sec_df['Top-1 %']]
    ax.bar(sec_df['Sector'], sec_df['Top-1 %'], color=clrs_s, alpha=0.85)
    ax.axhline(50, color='white', linestyle='--', linewidth=1, label='50% line')
    ax.set_ylabel('Top-1 Accuracy %')
    ax.set_title('Level 2 Top-1 Accuracy by Sector', fontweight='bold')
    ax.legend()
    for bar, v in zip(ax.patches, sec_df['Top-1 %']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v:.0f}%', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('data/modeling/per_sector.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
    plt.show()
    display(sec_df)

## 5. Scenario B — Consensus Output Distribution

In [ ]:
if 'cons_prob_1' in comp_df.columns:
    probs = comp_df['cons_prob_1'].dropna()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.hist(probs, bins=40, color=GREEN, alpha=0.85, edgecolor='#0F1117')
    ax.axvline(probs.mean(),   color=GOLD, linestyle='--', linewidth=2, label=f'Mean {probs.mean():.1%}')
    ax.axvline(probs.median(), color=RED,  linestyle='--', linewidth=2, label=f'Median {probs.median():.1%}')
    ax.set_title('Consensus Probability Distribution\n(Production has no equivalent)', fontweight='bold')
    ax.set_xlabel('Top-1 Consensus Probability')
    ax.set_ylabel('Companies')
    ax.legend()

    ax = axes[1]
    bands = {
        'HIGH (≥70%)': (probs >= 0.70).sum(),
        'MED (40-69%)': ((probs >= 0.40) & (probs < 0.70)).sum(),
        'LOW (<40%)': (probs < 0.40).sum(),
    }
    bars = ax.bar(list(bands.keys()), list(bands.values()),
                  color=[GREEN, GOLD, RED], alpha=0.85)
    for bar, (label, v) in zip(bars, bands.items()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{v:,} ({v/n:.0%})', ha='center', fontsize=10)
    ax.set_title('Confidence Bands', fontweight='bold')
    ax.set_ylabel('Companies')

    plt.tight_layout()
    plt.savefig('data/modeling/probability_dist.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
    plt.show()

## 6. UK SIC Gap — Available vs Used

In [ ]:
uk_avail = comp_df['gap_uk_sic_available'].notna().sum() if 'gap_uk_sic_available' in comp_df.columns else 0
uk_used  = comp_df['cons_uk_sic_usable'].sum() if 'cons_uk_sic_usable' in comp_df.columns else 0
uk_rest  = n - uk_avail

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'UK SIC 2007: {uk_avail} companies had UK SIC in OC response',
             fontsize=12, fontweight='bold')

ax = axes[0]
ax.pie([0, uk_avail, uk_rest],
       labels=[f'Stored (0)', f'Available, DROPPED ({uk_avail})', f'Not in OC ({uk_rest})'],
       colors=[GREEN, RED, GRAY], startangle=90,
       autopct=lambda p: f'{p:.1f}%' if p > 1 else '',
       textprops={'color':'#E2E8F0'})
ax.set_title('Scenario A — Production', fontweight='bold')

ax = axes[1]
ax.pie([uk_used, max(0, uk_avail - uk_used), uk_rest],
       labels=[f'Used as primary ({uk_used})', f'OC returned, low priority ({max(0,uk_avail-uk_used)})', f'Not in OC ({uk_rest})'],
       colors=[GREEN, GOLD, GRAY], startangle=90,
       autopct=lambda p: f'{p:.1f}%' if p > 1 else '',
       textprops={'color':'#E2E8F0'})
ax.set_title('Scenario B — Consensus', fontweight='bold')

plt.tight_layout()
plt.savefig('data/modeling/uk_sic_gap.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()
print(f"Production stored: 0 UK SIC codes")
print(f"Consensus uses:    {uk_used} UK SIC codes as primary output for GB companies")

## 7. AML / KYB Signals — What Consensus Produces That Production Cannot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# KYB distribution
ax = axes[0]
if 'cons_kyb' in comp_df.columns:
    kyb = comp_df['cons_kyb'].value_counts()
    order = ['APPROVE','REVIEW','ESCALATE','REJECT']
    vals = [kyb.get(k, 0) for k in order]
    clrs_k = [GREEN, BLUE, GOLD, RED]
    bars = ax.bar(order, vals, color=clrs_k, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{v:,} ({v/n:.0%})', ha='center', fontsize=10)
    ax.set_title('KYB Recommendation (Scenario B only)\nProduction: not produced', fontweight='bold')
    ax.set_ylabel('Companies')

# AML signal breakdown
ax = axes[1]
if 'cons_aml_flags' in comp_df.columns:
    all_flags = [f.strip() for row in comp_df['cons_aml_flags']
                 for f in str(row).split(';') if f.strip() and f.strip() != '—']
    fc = Counter(all_flags).most_common()
    if fc:
        labels_f = [x[0].replace('_','\n') for x in fc]
        values_f = [x[1] for x in fc]
        clrs_f = [RED if 'RISK' in l or 'DISC' in l else GOLD for l in labels_f]
        bars = ax.barh(labels_f, values_f, color=clrs_f, alpha=0.85)
        for bar, v in zip(bars, values_f):
            ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
                    f'{v:,} ({v/n:.0%})', va='center', fontsize=9)
        ax.set_xlim(0, max(values_f) * 1.25)
        ax.set_title('AML Signal Types (Scenario B only)\nProduction: 0 signals', fontweight='bold')
        ax.set_xlabel('Companies Flagged')

plt.tight_layout()
plt.savefig('data/modeling/aml_kyb.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 8. Code Agreement — Where Production and Consensus Differ

In [ ]:
if 'codes_agree' in comp_df.columns:
    agree = int(comp_df['codes_agree'].sum())
    disagree = n - agree

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    ax = axes[0]
    ax.pie([agree, disagree],
           labels=[f'Agree\n{agree:,} ({agree/n:.0%})',
                   f'Disagree\n{disagree:,} ({disagree/n:.0%})'],
           colors=[GREEN, RED], startangle=90,
           autopct='%1.1f%%', textprops={'color':'#E2E8F0'})
    ax.set_title('NAICS Code Agreement\nProduction vs Consensus', fontweight='bold')

    ax = axes[1]
    if '_sector' in comp_df.columns:
        dis_by_sec = comp_df[~comp_df['codes_agree']]['_sector'].value_counts().head(8)
        ax.barh(dis_by_sec.index, dis_by_sec.values, color=RED, alpha=0.8)
        for bar, v in zip(ax.patches, dis_by_sec.values):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                    str(v), va='center', fontsize=10)
        ax.set_title('Disagreements by Sector\n(Consensus overrides production rule)', fontweight='bold')
        ax.set_xlabel('Disagreements')

    plt.tight_layout()
    plt.savefig('data/modeling/code_agreement.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
    plt.show()

    print(f"Where they agree:    {agree:,} ({agree/n:.0%})")
    print(f"Where they disagree: {disagree:,} ({disagree/n:.0%})")
    print("→ Disagreements = cases where Consensus overrides the winner-takes-all rule")

## 9. Sample Output — 20 Companies Side by Side

In [ ]:
display_cols = [c for c in [
    'company_name', 'oc_jurisdiction_code',
    # Scenario A
    'prod_winning_src', 'prod_naics', 'prod_probability', 'gap_uk_sic_available',
    # Scenario B
    'cons_naics_1', 'cons_probability_str', 'cons_uk_sic', 'cons_aml_flags', 'cons_kyb',
    # Agreement
    'codes_agree',
] if c in comp_df.columns]

sample = comp_df[display_cols].sample(min(20, n), random_state=42)

def colour_agree(val):
    if val == True:  return 'background-color: #1a5c30; color: white'
    if val == False: return 'background-color: #5c1a1a; color: white'
    return ''

agree_cols = [c for c in ['codes_agree'] if c in sample.columns]
styled = sample.style
if agree_cols:
    styled = styled.applymap(colour_agree, subset=agree_cols)

display(styled.set_table_styles([
    {'selector': 'th', 'props': [('font-size','10px'),('background','#1A1F2E'),('color','#63B3ED')]},
    {'selector': 'td', 'props': [('font-size','10px')]},
]))

print("\nColumn guide:")
print("  prod_naics       = Production's single NAICS (no probability)")
print("  gap_uk_sic       = UK SIC available from OC but NOT stored (production gap)")
print("  cons_naics_1     = Consensus XGBoost top prediction")
print("  cons_probability = Calibrated confidence from Level 2 model")
print("  cons_uk_sic      = UK SIC now used as primary for GB companies")
print("  cons_aml_flags   = AML signals — none from Production")
print("  cons_kyb         = APPROVE / REVIEW / ESCALATE / REJECT")

## 10. Final Summary — The Full Delta

In [ ]:
engine = ComparisonEngine()
engine.comparison_df = comp_df
delta = engine.delta_table()
display(delta.style.set_table_styles([
    {'selector': 'th', 'props': [('background','#1A1F2E'),('color','#63B3ED'),('font-size','12px')]},
    {'selector': 'td', 'props': [('font-size','11px')]},
]))

print()
print("Note on accuracy with synthetic data:")
print("  Synthetic accuracy (~5-15%) reflects learning from confidence signals alone.")
print("  With real rel_business_industry_naics labels from case-service PostgreSQL,")
print("  expected Top-1: 55-75%  |  Top-3: 80-90%")
print()
print("What Consensus adds that Production never produces:")
print("  ✅ Calibrated probability (confidence)")
print("  ✅ Top-3 alternative codes")
print("  ✅ UK SIC / NACE / ISIC routed by jurisdiction")
print("  ✅ 5 AML/KYB risk signal types")
print("  ✅ APPROVE / REVIEW / ESCALATE / REJECT recommendation")
print("  ✅ Source lineage with weights per vendor")
print("  ✅ Jurisdiction-aware taxonomy routing")